# 🔢 [심화] MNIST로 0부터 9까지 숫자 판별하기

> ### 🗺️ 이 심화 실습에서 배우는 것
> 1. 기본 실습(1 vs 2 이진분류)을 0~9 다중분류로 확장하는 방법
> 2. 실제 손글씨 데이터셋 MNIST 불러오고 살펴보기
> 3. 한 줄 한 줄 이해하면서 숫자 판별 모델 만들고 평가하기

In [ ]:
# 한글을 그래프에 보여주는 라이브러리
!pip install -q koreanize_matplotlib
import koreanize_matplotlib

## 1. 실습 환경 준비

필요한 도구들을 불러옵니다. 기본 실습과 같은 도구들입니다.

In [ ]:
import numpy as np                      # 숫자 계산용 (행렬 다루기)
import matplotlib.pyplot as plt         # 그림 그리기용

plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지
print("✅ 준비 완료!")

## 2. MNIST 데이터셋 불러오기

**MNIST**는 손글씨 숫자 사진 모음집입니다.
- 사진 총 70,000장 (학습용 60,000장 + 시험용 10,000장)
- 각 사진은 28×28 픽셀 = 784개의 숫자
- 각 픽셀 값은 0(검정) ~ 255(흰색)
- 정답 라벨은 0부터 9까지

기본 실습의 6×6 손그림과 달리, **실제 사람들이 쓴** 숫자입니다.

In [ ]:
from sklearn.datasets import fetch_openml   # 공개 데이터셋을 가져오는 도구

# MNIST 데이터셋 내려받기 (처음 한 번만 시간이 걸립니다)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# 사진 데이터 (70,000장 × 784픽셀)
X = mnist.data
# 정답 라벨 (70,000개, 0~9 중 하나)
y = mnist.target.astype(int)   # 문자열로 오니까 정수로 바꿔주기

print(f"사진 개수: {X.shape[0]}장")
print(f"한 장당 픽셀 수: {X.shape[1]}개 (28×28)")
print(f"라벨 개수: {y.shape[0]}개")
print(f"라벨 종류: {sorted(set(y))}")

## 3. 사진 직접 살펴보기

숫자가 어떻게 생겼는지 눈으로 확인해봅시다.
784개의 숫자를 다시 28×28 모양으로 바꿔서 보여줍니다.

In [ ]:
# 0~9까지 각 숫자 하나씩 골라서 보여주기
fig, axes = plt.subplots(2, 5, figsize=(10, 4))   # 2행 5열로 10칸 만들기
axes = axes.ravel()                                # 2차원을 1차원으로 펴기

for digit in range(10):                            # 0부터 9까지 반복
    idx = np.where(y == digit)[0][0]               # 해당 숫자의 첫 번째 사진 찾기
    img = X[idx].reshape(28, 28)                    # 784개를 28×28 모양으로 바꾸기
    axes[digit].imshow(img, cmap='gray')            # 흑백으로 보여주기
    axes[digit].set_title(f"숫자 {digit}", fontsize=14)  # 제목 붙이기
    axes[digit].axis('off')                        # 축 눈금 끄기

plt.suptitle("MNIST 숫자 0~9", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 숫자별 개수 확인하기

0부터 9까지 골고루 들어있는지 확인합니다.
한쪽 숫자에 치우쳐 있으면 모델이 편향될 수 있거든요.

In [ ]:
# 각 숫자가 몇 장씩 있는지 세기
unique, counts = np.unique(y, return_counts=True)

# 막대 그래프로 보여주기
plt.figure(figsize=(10, 4))
plt.bar(unique, counts)                        # x=숫자, y=개수
plt.xlabel("숫자", fontsize=12)
plt.ylabel("사진 개수", fontsize=12)
plt.title("숫자별 사진 개수", fontsize=14, fontweight='bold')
plt.xticks(range(10))                          # x축에 0~9 표시
for i, c in enumerate(counts):                 # 각 막대 위에 숫자 쓰기
    plt.text(i, c + 100, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("→ 0~9가 대략 7,000장씩 골고루 들어있습니다!")

## 5. 정규화: 0~255를 0~1로 바꾸기

픽셀 값이 0~255로 크면 계산이 불안정해집니다.
255로 나누면 모든 값이 0~1 사이가 되어 학습이 안정적입니다.

**기본 실습 복습:** 6×6 실습에서도 같은 이유로 값을 다뤘죠?

In [ ]:
# 255로 나누어 0~1 사이 값으로 만들기
X = X / 255.0

print(f"정규화 후 가장 작은 값: {X.min():.2f}")
print(f"정규화 후 가장 큰 값: {X.max():.2f}")
print("→ 이제 모든 픽셀이 0~1 사이입니다!")

## 6. 학습용과 시험용 나누기

- **학습용 60,000장**: 모델이 공부하는 문제집
- **시험용 10,000장**: 한 번도 본 적 없는 실전 시험지

MNIST는 이미 순서대로 나뉘어 있습니다. 앞 60,000장이 학습용!

In [ ]:
# 앞 60,000장 = 학습용
X_train = X[:60000]
y_train = y[:60000]

# 뒤 10,000장 = 시험용
X_test = X[60000:]
y_test = y[60000:]

print(f"학습용: 사진 {X_train.shape[0]}장, 정답 {y_train.shape[0]}개")
print(f"시험용: 사진 {X_test.shape[0]}장, 정답 {y_test.shape[0]}개")

## 7. 모델 만들기: 1 vs 2와 뭐가 달라질까?

| | 기본 실습 (1 vs 2) | 심화 실습 (0~9) |
|---|---|---|
| 구분할 숫자 | 2개 | 10개 |
| 출력 뉴런 | 1개 (1이냐 아니냐) | 10개 (각 숫자일 확률) |
| 마지막 활성화 | 시그모이드 | 소프트맥스 |

**소프트맥스**는 10개의 점수를 "확률"로 바꿔줍니다.
전부 더하면 1이 되고, 가장 높은 게 모델의 답입니다.

여기서는 개념 이해에 집중하기 위해 `MLPClassifier`(신경망)를 씁니다.
은닉층 1개, 간단한 구조로도 97% 이상 맞힙니다.

In [ ]:
from sklearn.neural_network import MLPClassifier   # 신경망 모델 도구

# 모델 만들기 - 각 옵션을 한 줄씩 봅시다
model = MLPClassifier(
    hidden_layer_sizes=(128,),   # 은닉층: 뉴런 128개짜리 1층
    activation='relu',            # 은닉층 활성화 함수: 렐루 (0 이하는 0, 양수는 그대로)
    solver='adam',                # 학습 방법: 아담 (똑똑한 경사하강법)
    max_iter=20,                  # 전체 데이터를 20번 반복 학습
    random_state=42,               # 난수 고정 (매번 같은 결과 나오게)
    verbose=True                  # 학습 과정을 화면에 보여주기
)

print("✅ 모델 준비 완료!")
print("   입력: 784개 픽셀 → 은닉층: 128개 뉴런 → 출력: 10개 (0~9 확률)")

## 8. 학습시키기

60,000장으로 공부시킵니다. 1~2분 정도 걸립니다.
아래에 점수가 점점 올라가는 게 보일 거예요.

In [ ]:
# 학습 시작! (시간이 조금 걸립니다)
model.fit(X_train, y_train)

print("\n✅ 학습 완료!")

## 9. 시험 보기: 한 번도 못 본 10,000장

진짜 실력은 **처음 보는 문제**로 평가합니다.

In [ ]:
# 시험용 10,000장으로 점수 매기기
score = model.score(X_test, y_test)

print(f"🎯 시험 점수(정확도): {score*100:.2f}%")
print(f"   → 10,000장 중 {int(score*10000)}장을 맞혔습니다!")

## 10. 모델의 답 직접 확인하기

시험지 10장을 꺼내서 모델이 뭐라고 답했는지 봅시다.
초록색 = 정답, 빨간색 = 오답입니다.

In [ ]:
# 시험용에서 10장 골라서 예측해보기
n_show = 10
X_show = X_test[:n_show]          # 사진 10장
y_true = y_test[:n_show]          # 진짜 정답 10개
y_pred = model.predict(X_show)    # 모델의 답 10개

# 그림으로 보여주기
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(n_show):
    img = X_show[i].reshape(28, 28)              # 28×28로 되돌리기
    axes[i].imshow(img, cmap='gray')             # 보여주기
    correct = (y_pred[i] == y_true[i])           # 맞았는지 확인
    color = 'green' if correct else 'red'       # 맞으면 초록, 틀리면 빨강
    mark = '⭕' if correct else '❌'
    axes[i].set_title(f"{mark} 예측:{y_pred[i]} 정답:{y_true[i]}",
                      fontsize=12, color=color, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("모델의 예측 결과", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. 틀린 문제는 어떤 게 있을까?

모델이 헷갈린 사진들을 모아봅시다.
사람도 헷갈릴 만한 글씨인지 확인해보세요!

In [ ]:
# 전체 시험지에서 틀린 것만 골라내기
y_all_pred = model.predict(X_test)              # 10,000장 전부 예측
wrong_idx = np.where(y_all_pred != y_test)[0]   # 틀린 것들의 번호

print(f"틀린 개수: {len(wrong_idx)}장 / 10,000장")

# 틀린 것 중 10장 보여주기
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    idx = wrong_idx[i]                          # 틀린 사진의 번호
    img = X_test[idx].reshape(28, 28)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"예측:{y_all_pred[idx]} 정답:{y_test[idx]}",
                      fontsize=12, color='red', fontweight='bold')
    axes[i].axis('off')

plt.suptitle("모델이 틀린 사진들", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("→ 사람이 봐도 헷갈리는 글씨가 많죠?")

## 🎓 정리: 기본 실습과의 연결

### 무엇이 같고 무엇이 다른가?

1. **같은 것**: 사진을 숫자로 보고, 가중치를 곱하고, 활성화 함수로 답을 내는 원리
2. **다른 것**:
   - 사진 크기: 6×6(36픽셀) → 28×28(784픽셀)
   - 구분 개수: 2개 → 10개
   - 출력층: 뉴런 1개(시그모이드) → 뉴런 10개(소프트맥스)
   - 데이터: 손으로 그린 2장 → 실제 손글씨 70,000장

### 🤔 생각해보기
1. 은닉층 뉴런을 128개에서 32개로 줄이면 정확도가 어떻게 될까요?
2. 학습 반복 횟수(max_iter)를 5번으로 줄이면 어떻게 될까요?
3. 3과 8처럼 비슷하게 생긴 숫자는 왜 더 많이 틀릴까요?

### 💡 다음 단계
- CNN(합성곱)을 쓰면 99%까지 올라갑니다. 기본 실습에서 배운 합성곱을 여기에 적용해보세요!
- 내 손글씨를 직접 써서 테스트해보는 것도 재밌습니다.